# HTAN — Hyper TransAttUNet
### GlaS — Gland Segmentation Experiments
---

## 0. Setup

In [ ]:
import os
import sys
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import torch

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f"Project root : {PROJECT_ROOT}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")

---
# 1. GlaS — Gland Segmentation

## 1.0 Download Data

GlaS requires manual download from Warwick QU:
https://warwick.ac.uk/fac/cross_fac/tia/data/glascontest/download/

Extract to `/opt/dlami/nvme/HTAN/data/glas/` with `train/` and `test/` subdirectories.

## 1.1 Verify Data

In [ ]:
TRAIN_DIR = Path("/opt/dlami/nvme/HTAN/data/glas/train")
TEST_DIR  = Path("/opt/dlami/nvme/HTAN/data/glas/test")

train_imgs  = sorted([f for f in TRAIN_DIR.glob("*.bmp") if "anno" not in f.name])
train_masks = sorted([f for f in TRAIN_DIR.glob("*.bmp") if "anno" in f.name])
test_imgs   = sorted([f for f in TEST_DIR.glob("*.bmp")  if "anno" not in f.name])
test_masks  = sorted([f for f in TEST_DIR.glob("*.bmp")  if "anno" in f.name])

print(f"Train images : {len(train_imgs)}")
print(f"Train masks  : {len(train_masks)}")
print(f"Test images  : {len(test_imgs)}")
print(f"Test masks   : {len(test_masks)}")
print(f"Match train  : {'YES' if len(train_imgs) == len(train_masks) else 'NO — CHECK THIS'}")
print(f"Match test   : {'YES' if len(test_imgs) == len(test_masks) else 'NO — CHECK THIS'}")
# Expected: Train 85, Test 80

In [ ]:
# 3 random samples
samples = random.sample(train_imgs, 3)
fig, axes = plt.subplots(3, 2, figsize=(10, 12))
fig.suptitle("GlaS Sample Images", fontsize=14, fontweight="bold")

for i, img_path in enumerate(samples):
    mask_path = img_path.parent / f"{img_path.stem}_anno.bmp"
    axes[i, 0].imshow(Image.open(img_path).convert("RGB"))
    axes[i, 0].set_title(f"Input: {img_path.name}")
    axes[i, 0].axis("off")
    axes[i, 1].imshow(Image.open(mask_path).convert("L"), cmap="gray")
    axes[i, 1].set_title(f"Mask: {mask_path.name}")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

## 1.2 Sanity Check — Model Forward Pass

GlaS uses **128×128** images — different from ISIC (256×256).
All HTAN models must be instantiated with `img_size=128`.

In [ ]:
from models.transattunet.TransAttUnet import TransAttUNet_R
from models.baselines.unet import UNet
from models.baselines.doubleunet import DoubleUNet
from models.htan.htan import HTAN_1, HTAN_2, HTAN_1_Hres_only

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
dummy  = torch.randn(2, 3, 128, 128).to(DEVICE)

models_to_check = {
    "unet":             UNet(),
    "doubleunet":       DoubleUNet(),
    "transattunet":     TransAttUNet_R(),
    "htan_1_n2":        HTAN_1(expansion_n=2, img_size=128),
    "htan_1_n4":        HTAN_1(expansion_n=4, img_size=128),
    "htan_2_n2":        HTAN_2(expansion_n=2, img_size=128),
    "htan_1_hres_only": HTAN_1_Hres_only(expansion_n=4, img_size=128),
}

print(f"{'Model':<22} {'Params':>12} {'Output':>15} {'Status'}")
print("-" * 60)

for name, model in models_to_check.items():
    try:
        model = model.to(DEVICE).eval()
        with torch.no_grad():
            out = model(dummy)
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"{name:<22} {n_params:>12,} {str(tuple(out.shape)):>15}   OK")
    except Exception as e:
        print(f"{name:<22} ERROR — {str(e)[:50]}")
    finally:
        del model
        torch.cuda.empty_cache()

## 1.3 Training

Each cell trains one model independently. Resume is automatic if interrupted.

In [ ]:
# 1.3.1 U-Net
!python3 train.py --model unet --dataset glas

In [ ]:
# 1.3.2 DoubleU-Net
!python3 train.py --model doubleunet --dataset glas

In [ ]:
# 1.3.3 TransAttUNet_R
!python3 train.py --model transattunet --dataset glas

In [ ]:
!python3 evaluate.py --model transattunet --dataset glas

In [ ]:
# 1.3.4 HTAN_1 n=2
!python3 train.py --model htan_1_n2 --dataset glas

In [ ]:
!python3 evaluate.py --model htan_1_n2 --dataset glas

In [ ]:
# 1.3.5 HTAN_1 n=4
!python3 train.py --model htan_1_n4 --dataset glas

In [ ]:
!python3 evaluate.py --model htan_1_n4 --dataset glas

In [ ]:
# 1.3.6 HTAN_2 n=2
!python3 train.py --model htan_2_n2 --dataset glas

In [ ]:
!python3 evaluate.py --model htan_2_n2 --dataset glas

In [ ]:
# 1.3.7 HTAN_1 Hres-only (ablation)
!python3 train.py --model htan_1_hres_only --dataset glas

In [ ]:
!python3 evaluate.py --model htan_1_hres_only --dataset glas

## 1.4 Evaluate All Models

In [ ]:
!python3 evaluate.py --model all --dataset glas

## 1.5 Results Table

In [ ]:
RESULTS_DIR = Path("/opt/dlami/nvme/HTAN/results/glas")
SAVES_ROOT  = Path("/opt/dlami/nvme/HTAN/saves")

# Paper numbers — Table V (TransAttUNet paper)
PAPER_RESULTS = {
    "U-Net†":          {"dice": 75.73, "iou": 91.03, "acc": None,  "rec": None,  "pre": None},
    "ResUNet†":        {"dice": 80.88, "iou": 69.11, "acc": 81.49, "rec": 85.11, "pre": 80.01},
    "Swin-UNet†":      {"dice": 86.70, "iou": 77.32, "acc": None,  "rec": 89.00, "pre": 86.12},
    "SegFormer†":      {"dice": 87.36, "iou": 79.71, "acc": None,  "rec": 85.56, "pre": 86.53},
    "TransAttUNet_R†": {"dice": 89.11, "iou": 81.13, "acc": 89.02, "rec": 90.08, "pre": 88.95},
}

MODEL_LABELS = {
    "unet":             "U-Net*",
    "doubleunet":       "DoubleU-Net*",
    "transattunet":     "TransAttUNet_R*",
    "htan_1_n2":        "HTAN_1 n=2 (Ours)",
    "htan_1_n4":        "HTAN_1 n=4 (Ours)",
    "htan_2_n2":        "HTAN_2 n=2 (Ours)",
    "htan_1_hres_only": "HTAN_1 Hres-only (Ours)",
}

OUR_RESULTS = {}
for key, label in MODEL_LABELS.items():
    path = RESULTS_DIR / f"{key}.json"
    if path.exists():
        with open(path) as f:
            data = json.load(f)
        OUR_RESULTS[label] = {k: data[k] for k in ["dice", "iou", "acc", "rec", "pre"]}
    else:
        print(f"Not trained yet: {key}")

def fmt(v):
    return f"{v:.2f}" if v is not None else "—"

ALL = {**PAPER_RESULTS, **OUR_RESULTS}
print(f"\n{'Method':<26} {'Dice':>6} {'IoU':>6} {'ACC':>6} {'REC':>6} {'PRE':>6}")
print("-" * 58)
for name, m in ALL.items():
    print(f"{name:<26} {fmt(m['dice']):>6} {fmt(m['iou']):>6} "
          f"{fmt(m['acc']):>6} {fmt(m['rec']):>6} {fmt(m['pre']):>6}")

## 1.6 Training Curves

In [ ]:
def load_history(model_name, dataset="glas"):
    path = SAVES_ROOT / f"{model_name}_{dataset}" / "resume_checkpoint.pth"
    if not path.exists():
        print(f"No checkpoint: {model_name}_{dataset}")
        return None
    return torch.load(path, map_location="cpu")["history"]

def plot_metric(histories, metric, title):
    plt.figure(figsize=(10, 5))
    for name, h in histories.items():
        if h and metric in h:
            plt.plot(h[metric], label=name)
    plt.xlabel("Epoch")
    plt.ylabel(metric.capitalize())
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

MODELS   = list(MODEL_LABELS.keys())
histories = {m: load_history(m) for m in MODELS}

plot_metric(histories, "dice",       "1.6.1 Validation Dice — GlaS")
plot_metric(histories, "iou",        "1.6.2 Validation IoU — GlaS")
plot_metric(histories, "train_loss", "1.6.3 Training Loss — GlaS")
plot_metric(histories, "val_loss",   "1.6.4 Validation Loss — GlaS")

## 1.7 Ablation Table

In [ ]:
ABLATION = [
    ("transattunet",     "—", "—", "none",  "TransAttUNet_R (baseline)"),
    ("htan_1_hres_only", "1", "4", "H_res", "HTAN_1 Hres-only"),
    ("htan_1_n2",        "1", "2", "full",  "HTAN_1 n=2"),
    ("htan_1_n4",        "1", "4", "full",  "HTAN_1 n=4"),
    ("htan_2_n2",        "2", "2", "full",  "HTAN_2 n=2"),
]

print(f"{'Model':<26} {'Blocks':>6} {'n':>4} {'Mappings':>10} {'Dice':>6} {'IoU':>6}")
print("-" * 62)

for key, blocks, n_val, mappings, name in ABLATION:
    path = RESULTS_DIR / f"{key}.json"
    if path.exists():
        with open(path) as f:
            d = json.load(f)
        dice, iou = f"{d['dice']:.2f}", f"{d['iou']:.2f}"
    else:
        dice = iou = "—"
    print(f"{name:<26} {blocks:>6} {n_val:>4} {mappings:>10} {dice:>6} {iou:>6}")

## 1.8 Visual Predictions

In [ ]:
from datasets.glas_dataset import get_loaders

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_best(model, model_name, dataset="glas"):
    path = SAVES_ROOT / f"{model_name}_{dataset}" / "best_model.pth"
    if not path.exists():
        print(f"No best model: {model_name}_{dataset}")
        return None
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    return model.to(DEVICE).eval()

_, test_loader = get_loaders(img_size=128, batch_size=4)
imgs, masks    = next(iter(test_loader))
imgs_gpu       = imgs.to(DEVICE)

tan  = load_best(TransAttUNet_R(), "transattunet")
htan = load_best(HTAN_2(expansion_n=2, img_size=128), "htan_2_n2")

with torch.no_grad():
    pred_tan  = (torch.sigmoid(tan(imgs_gpu))  > 0.5).cpu() if tan  else None
    pred_htan = (torch.sigmoid(htan(imgs_gpu)) > 0.5).cpu() if htan else None

def denorm(t):
    return (t * 0.5 + 0.5).clamp(0, 1)

n_show = min(4, imgs.shape[0])
fig, axes = plt.subplots(n_show, 4, figsize=(16, 4 * n_show))
fig.suptitle("1.8 Visual Predictions — GlaS", fontsize=14, fontweight="bold")

for col, title in enumerate(["Input", "Ground Truth", "TransAttUNet_R", "HTAN_2 n=2"]):
    axes[0, col].set_title(title, fontweight="bold")

for i in range(n_show):
    axes[i, 0].imshow(denorm(imgs[i]).permute(1, 2, 0).numpy())
    axes[i, 1].imshow(masks[i, 0].numpy(), cmap="gray")
    axes[i, 2].imshow(pred_tan[i, 0].numpy()  if pred_tan  is not None else np.zeros((128,128)), cmap="gray")
    axes[i, 3].imshow(pred_htan[i, 0].numpy() if pred_htan is not None else np.zeros((128,128)), cmap="gray")
    for ax in axes[i]: ax.axis("off")

plt.tight_layout()
plt.show()